# Part 5C — Agentic RAG (Notebook 07)

This notebook implements adaptive routing over retrieval tools.


## Tutorial Goals

This notebook is a standalone, zero-to-hero tutorial with:

1. Concept explanation from first principles
2. Architecture and workflow breakdown
3. End-to-end implementation code
4. Real execution outputs and benchmark metrics
5. Practical analysis and production takeaways


## What is this technique?

        ### Definition and core concepts
        Agentic RAG uses decision logic to select actions based on query and intermediate quality signals.

        ### Why was this technique developed?
        Fixed pipelines apply the same path to all queries, which is inefficient and brittle.

        ### What limitations of traditional RAG does it solve?
        It reduces one-size-fits-all behavior and enables adaptive retrieval choices.

        ### Architecture and workflow diagram explanation

```mermaid
graph TD
    Q[Query] --> Router[Route Decision]
    Router --> H[Hybrid]
    Router --> G[Graph]
    Router --> B[BM25]
    H --> Judge[Retrieval Judge]
    G --> Judge
    B --> Judge
    Judge --> Gen[Generation]
```


        ### Component-by-component breakdown
        Router policy, retrieval toolset, quality judge, generation, trace logging.

        ### When should it be used in real-world systems?
        Use when user queries vary significantly and reliability requires adaptive behavior.

        ### Advantages and disadvantages
        **Advantages**
        - Adaptive retrieval path selection
- Better observability
- Easier targeted optimization

        **Disadvantages**
        - Additional orchestration complexity
- More calls and latency variance

        ### Comparison against standard RAG and other implemented RAG variants
        Compared with GraphRAG, Agentic RAG adds dynamic control. Compared with CRAG, this version is less corrective but simpler.

        ### Implementation details and design decisions used in this project
        This implementation uses deterministic routing plus granite-based judges and captures full execution traces.


In [ ]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rag_v2.data import load_base_corpus, load_papers_from_chunks
from src.rag_v2.retrieval import DenseRetriever, BM25Retriever, HybridRetriever
from src.rag_v2.metrics import build_keyword_eval_set, save_json
from src.rag_v2.graph import build_paper_entity_graph, build_entity_to_papers, expand_with_graph
from src.rag_v2.agentic import route_query, llm_answer, llm_faithfulness

ART = PROJECT_ROOT / 'artifacts' / 'rag_v2'
ART.mkdir(parents=True, exist_ok=True)

index, chunks = load_base_corpus()
papers = load_papers_from_chunks(chunks)

display(Markdown(f"Loaded **{len(chunks):,} chunks** from **{len(papers):,} papers** (FAISS dim={index.d})."))


In [ ]:
dense = DenseRetriever(index=index, chunks=chunks)
bm25 = BM25Retriever(chunks=chunks)
hybrid = HybridRetriever(dense=dense, bm25=bm25, alpha=0.7)

graph, paper_to_entities = build_paper_entity_graph(papers, max_entities_per_paper=8)
entity_to_papers = build_entity_to_papers(paper_to_entities)

paper_to_chunks = {}
for c in chunks:
    paper_to_chunks.setdefault(c['paper_id'], []).append(c)


def graph_retrieve(query: str, k: int = 8):
    seed = hybrid.retrieve(query, k=6)
    seed_papers = list(dict.fromkeys(r['paper_id'] for r in seed))
    extra = expand_with_graph(seed_papers, paper_to_entities, entity_to_papers, max_extra_papers=8)
    merged = list(seed)
    for pid in extra:
        for c in paper_to_chunks.get(pid, [])[:1]:
            merged.append({**c, 'score': 0.05, 'retriever': 'graph_expand'})
    merged.sort(key=lambda x: x.get('score', 0.0), reverse=True)
    seen, out = set(), []
    for row in merged:
        if row['chunk_id'] in seen:
            continue
        out.append(row)
        seen.add(row['chunk_id'])
        if len(out) >= k:
            break
    return out


def cheap_relevance_grade(question: str, docs: list[dict]) -> tuple[str, float]:
    tokens = [t for t in question.lower().split() if len(t) > 3]
    if not docs:
        return 'irrelevant', 0.0
    joined = ' '.join(d.get('text', '')[:500].lower() for d in docs[:3])
    overlap = sum(1 for t in set(tokens) if t in joined)
    conf = overlap / max(len(set(tokens)), 1)
    if conf >= 0.35:
        grade = 'relevant'
    elif conf >= 0.15:
        grade = 'partially_relevant'
    else:
        grade = 'irrelevant'
    return grade, round(float(conf), 3)


def run_agent(question: str, run_llm_checks: bool = False):
    t0 = time.perf_counter()
    route = route_query(question)
    if route == 'graph':
        docs = graph_retrieve(question, k=8)
    elif route == 'bm25':
        docs = bm25.retrieve(question, k=8)
    else:
        docs = hybrid.retrieve(question, k=8)

    grade, conf = cheap_relevance_grade(question, docs)

    if run_llm_checks:
        answer = llm_answer(question, [d['text'] for d in docs], model='granite4.1:8b')
        faith, reason = llm_faithfulness(question, answer, [d['text'] for d in docs], judge_model='granite4.1:8b')
    else:
        answer = 'Runtime-light row: routing and retrieval executed end to end; LLM generation is sampled for one row.'
        faith = np.nan
        reason = 'not_evaluated_in_light_mode'

    return {
        'question': question,
        'route': route,
        'retrieval_grade': grade,
        'retrieval_conf': conf,
        'faithfulness': faith,
        'faith_reason': reason,
        'llm_evaluated': run_llm_checks,
        'latency_ms': (time.perf_counter() - t0) * 1000,
    }


eval_set = build_keyword_eval_set(papers)[:6]
runs = []
for i, row in enumerate(eval_set):
    runs.append(run_agent(row['question'], run_llm_checks=(i == 0)))

agent_df = pd.DataFrame(runs)
agent_df


In [ ]:
route_stats = agent_df['route'].value_counts().to_dict()
faith_rows = agent_df['faithfulness'].dropna()

quality = {
    'faithfulness_mean': round(float(faith_rows.mean()), 4) if not faith_rows.empty else None,
    'llm_evaluated_rows': int(agent_df['llm_evaluated'].sum()),
    'latency_p50_ms': round(float(agent_df['latency_ms'].quantile(0.50)), 2),
    'latency_p95_ms': round(float(agent_df['latency_ms'].quantile(0.95)), 2),
}


def clean_nan(v):
    return None if isinstance(v, float) and np.isnan(v) else v

runs_clean = [{k: clean_nan(v) for k, v in row.items()} for row in runs]

out_json = ART / 'agentic' / '07_agentic_metrics.json'
out_json.parent.mkdir(parents=True, exist_ok=True)
save_json(out_json, {'route_stats': route_stats, 'quality': quality, 'runs': runs_clean})

route_stats, quality


In [ ]:
latency_gap = quality['latency_p95_ms'] - quality['latency_p50_ms']

analysis = (
    "## Post-run Analysis (Real Results)\n\n"
    f"- Route distribution: **{route_stats}**\n"
    f"- LLM-evaluated rows: **{quality['llm_evaluated_rows']}**\n"
    f"- Mean faithfulness (evaluated rows only): **{quality['faithfulness_mean']}**\n"
    f"- P50 latency: **{quality['latency_p50_ms']:.2f} ms**\n"
    f"- P95 latency: **{quality['latency_p95_ms']:.2f} ms**\n"
    f"- Latency spread (P95-P50): **{latency_gap:.2f} ms**\n\n"
    "- Observation: Most rows used lightweight routing-only evaluation, while sampled LLM rows dominate tail latency."
)
display(Markdown(analysis))
